# **Mapping short-let pressure in Bristol: A neighbourhood analysis of Airbnb listings**

# ML objectives 

# Change working directory

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\chheb\\OneDrive\\Documents\\Programming\\Mapping_Short_Let-Pressure_in_Bristol\\jupyter_notebooks'

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\chheb\\OneDrive\\Documents\\Programming\\Mapping_Short_Let-Pressure_in_Bristol'

# Import transformed data

In [4]:
from pathlib import Path  
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, GroupKFold, cross_validate
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)


RANDOM_STATE = 42


In [5]:
df = pd.read_csv("data/processed/listings_clean.csv", dtype={"id": str},
                 parse_dates=["first_review", "last_review"])
active = df[df["is_active"]].copy()
print(df.shape, active.shape) 

(2932, 51) (2561, 51)


# 1. Split the data

**Target:** 

`is_operator` (host with 3+ listings), on active listings.

**Features:** 

- `accommodates`, `price`, `is_entire_home`, `availability_365`, `review_scores_rating`, `number_of_reviews`, `host_is_superhost`. 
- Host-count columns are excluded because they define the target. 
- `beds` and `bedrooms` are dropped because they correlate 0.85-0.90 with `accommodates` (V8). - `minimum_nights` is dropped because H2 found a negligible effect that was not robust to removing the largest host. 
- Neighbourhood is not used in this first model, so the model cannot use geography, even though H1 found it matters.

**Split:** 

80/20, grouped by host so that no host appears in both sets. Otherwise listings from the same host would leak patterns between training and testing and inflate the test score. The test set is used once, at the end.

**Baseline:** 
A classifier that always predicts the larger group ("casual"). Both models must beat it to show they learned anything.

**Missing values:** L

These are left as they are here as imputation and scaling happen inside the pipeline. Fitted on training data only.

In [6]:

target = "is_operator"
features = ["accommodates", "price", "is_entire_home", "availability_365",
            "review_scores_rating", "number_of_reviews", "host_is_superhost"]

X = active[features].astype(float)
y = active[target].astype(int)
groups = active["host_key"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(X_train.shape, X_test.shape)
print("Operator share  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))
print("Hosts in both sets:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))   # expect 0
print(X_train.isna().sum())

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline accuracy on test:", round(baseline.score(X_test, y_test), 3))

(1971, 7) (590, 7)
Operator share  train: 0.397  test: 0.495
Hosts in both sets: 0
accommodates              0
price                   240
is_entire_home            0
availability_365          0
review_scores_rating    179
number_of_reviews         0
host_is_superhost         0
dtype: int64
Baseline accuracy on test: 0.505


In [7]:
top_host = active["host_key"].value_counts().index[0]
print("Largest host in test set:", top_host in set(groups.iloc[test_idx]))
print("Their listings in test:", (groups.iloc[test_idx] == top_host).sum(), "of", len(test_idx))

Largest host in test set: False
Their listings in test: 0 of 590


In [8]:
test_hosts = groups.iloc[test_idx]
print("Hosts in test set:", test_hosts.nunique())
print(test_hosts.value_counts().head(5))

Hosts in test set: 295
host_key
285    65
564    33
534    25
670    21
787    19
Name: count, dtype: int64


**Result:** 

The split gives 1,971 training and 590 test listings (23% in test; a group split cannot hit exactly 20%). No host appears in both sets.

**Class balance:** 

The operator share is 39.7% in training and 49.5% in testing, against 42% across all active listings. This difference comes from the host-grouped split because operators come in blocks. This means that which hosts land in the test set moves the share. I did not change the random seed to even it out. The largest host (127 listings) is in the training set, so it does not affect the test score.

**Baseline:** 

A classifier that always predicts "casual" scores 50.5% accuracy on the test set (about 58% across all active listings). This is the bar both models must beat. It is a low bar because the test set is close to balanced, so model comparisons also use cross-validation on the training data and per-group precision and recall, not accuracy alone.

**Missing values (training set):** 

`price` is missing for 240 listings (12%) and `review_scores_rating` for 179 (9%). They are imputed inside the pipeline, fitted on training data only, with an indicator for missingness, because a missing price or rating may itself carry information.

**Limitations:** 

The test set has 590 listings but far fewer independent hosts, so the test score is uncertain and a difference of a few points between models is not meaningful. 

The test set contains 295 hosts, but the five largest hold 28% of its listings (the largest has 65), and they provide over half of its operator listings. So the operator side of the test score depends heavily on a handful of hosts. Because the largest host is in the training data, a robustness check refits the models without it.

# 2. Building the pipeline


**Models:** 

- Logistic regression (an interpretable linear model)
- Decision tree (which can capture non-linear patterns and interactions) 
- Compared with a baseline that always predicts "casual".

**Preprocessing:** 
- Missing `price` and `review_scores_rating` values are filled with the median, with a missing-value indicator added. For the logistic regression
-  `price` and `number_of_reviews` are log-transformed (both are strongly skewed) and all numeric inputs are scaled. The tree needs neither. 
- All preprocessing sits inside a pipeline, so it is fitted on training data only and no information leaks from held-out data.

**Evaluation:** 
- 5-fold cross-validation on the training set, grouped by host so that no host appears in both a training and a validation fold. 
- Accuracy, precision, recall and F1 for the operator class are reported as mean ± standard deviation across folds. The test set is not used until the final model is chosen.



#### 1: Define the input groups

Inputs are grouped by how they need to be treated. `price` and `number_of_reviews` are strongly skewed (V0), so they are log-transformed for the logistic regression. The other numeric inputs are only scaled. The two yes/no inputs are already 0/1 so do not need to be changed.

In [9]:
log_cols = ["price", "number_of_reviews"]                                
num_cols = ["accommodates", "availability_365", "review_scores_rating"]
bin_cols = ["is_entire_home", "host_is_superhost"]

#### Step 2: Preprocessing for the logistic regression

Logistic regression is sensitive to the size and skew of its inputs, so each numeric column is (in order): log-transformed if skewed, missing values filled with the median, then scaled to mean 0 and standard deviation 1. `add_indicator=True` adds a 0/1 column for any input that had gaps (price and rating), because a missing value may itself carry information. Everything sits inside a pipeline, so medians and scaling are learned from training data only.

In [10]:
lr_prep = ColumnTransformer([
    ("log", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                      ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("scale", StandardScaler())]), log_cols),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("scale", StandardScaler())]), num_cols),
    ("bin", "passthrough", bin_cols),
], verbose_feature_names_out=False)

#### Step 3: Preprocessing for the decision tree

A decision tree splits on the order of values, so scaling and log transforms make no difference and are skipped. It only needs the gaps filled; the same median fill and missing-value indicator are used for the two columns with gaps, and every other column is passed through.

In [11]:
tree_prep = ColumnTransformer(
    [("imp", SimpleImputer(strategy="median", add_indicator=True), ["price", "review_scores_rating"])],
    remainder="passthrough", verbose_feature_names_out=False)

#### Step 4: Check the preprocessing

Before modelling, we need to confirm that the preprocessing does what is intended. In particular, we want to see that no missing values remain, and the missing-value indicator columns have been added. This fit is just an inspection so only uses the training set only.

In [12]:
X_lr = lr_prep.fit_transform(X_train)
print("Logistic inputs:", X_lr.shape, "| missing values left:", int(np.isnan(X_lr).sum()))
print(list(lr_prep.get_feature_names_out()))

X_tree = tree_prep.fit_transform(X_train)
print("Tree inputs:", X_tree.shape, "| missing values left:", int(np.isnan(X_tree).sum()))

Logistic inputs: (1971, 9) | missing values left: 0
['price', 'number_of_reviews', 'missingindicator_price', 'accommodates', 'availability_365', 'review_scores_rating', 'missingindicator_review_scores_rating', 'is_entire_home', 'host_is_superhost']
Tree inputs: (1971, 9) | missing values left: 0


#### Step 5: Assemble the models

Each model is bundled with its preprocessing into one pipeline. A baseline that always predicts "casual" is included so that the real models have something to beat. 

The decision tree starts at depth 4 (shallow enough to read). We will tune its depth and the regularisation strength of the logistic regression in a later step.

In [13]:
models = {
    "Baseline (always casual)": Pipeline([("model", DummyClassifier(strategy="most_frequent"))]),
    "Logistic regression": Pipeline([("prep", lr_prep),
                                     ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    "Decision tree (depth 4)": Pipeline([("prep", tree_prep),
                                         ("model", DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE))]),
}

#### Step 6: Set up grouped cross-validation

The models are compared with 5-fold cross-validation on the training set only. Folds are grouped by host, so no host appears in both a training and a validation fold, matching the train/test split. 

The pipelines are refitted inside each fold, so preprocessing never sees the validation data. Accuracy, precision, recall and F1 are reported for the operator class.

In [14]:
groups_train = groups.iloc[train_idx]
cv = GroupKFold(n_splits=5)
scoring = ["accuracy", "precision", "recall", "f1"]

#### Step 7: Compare the models

Mean and standard deviation across the five folds. The spread shows how much each result depends on which hosts fall in which fold; differences smaller than the spread are not meaningful.

In [15]:
rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, groups=groups_train, scoring=scoring)
    rows.append({"model": name,
                 **{m: f"{res['test_' + m].mean():.3f} ± {res['test_' + m].std():.3f}" for m in scoring}})
cv_results = pd.DataFrame(rows)
print(cv_results.to_string(index=False))

c:\Users\chheb\OneDrive\Documents\Programming\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\chheb\OneDrive\Documents\Programming\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\chheb\OneDrive\Documents\Programming\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

                   model      accuracy     precision        recall            f1
Baseline (always casual) 0.603 ± 0.006 0.000 ± 0.000 0.000 ± 0.000 0.000 ± 0.000
     Logistic regression 0.628 ± 0.030 0.558 ± 0.070 0.272 ± 0.086 0.362 ± 0.087
 Decision tree (depth 4) 0.661 ± 0.020 0.572 ± 0.020 0.561 ± 0.087 0.564 ± 0.055


**Result:** 

Both models beat the baseline of always predicting "casual" (60.3% accuracy), but by different margins.

The decision tree scores best (accuracy 0.661 ± 0.020, precision 0.572, recall 0.561, F1 0.564). 

The logistic regression is only 2.5 points above the baseline on accuracy (0.628 ± 0.030) and finds just 27% of operator listings (recall 0.272, F1 0.362).

**Interpretation:** 

The tree's advantage is clear on recall and F1 (the gap is larger than the fold-to-fold spread) but not on accuracy (3 points, within the spread). 

A possible reason for is that the tree can use non-linear patterns such as the U-shaped availability distribution (V4), which the linear model cannot. But this was not tested. The logistic regression's low recall also reflects its default decision threshold, so tuning (class weighting) may narrow the gap. Overall, listing characteristics separate operator and casual listings only modestly, consistent with the weak correlations in V8.

**Limitations:** 

Results are cross-validated on the training set only; the test set has not been used. Folds contain few large hosts, so the spread is rough and likely to understate the real uncertainty.

# 3. Tuning the model

In [16]:
tree_grid = {"model__max_depth": [2, 3, 4, 5, 6, 8],
             "model__min_samples_leaf": [5, 20, 50]}

lr_grid = {"model__C": [0.01, 0.1, 1, 10],
           "model__class_weight": [None, "balanced"]}